### Extract and Prepare Mock CSV data from YFinance  
- Download historical stock price data for major tech companies (AAPL, MSFT, etc.) from Yahoo Finance using the yfinance library. 
- Fetche daily prices between January 2020 and June 2025, then processes the multi-level column structure into a clean, long-format DataFrame. 
- Creates a 'data' directory and saves the final output as a CSV file with columns for Date, Ticker, and various price metrics (Open, High, Low, Close, etc.), making it ready for analysis. 
- Transform data as CSV.

In [ ]:
import yfinance as yf
import pandas as pd
import os
import pandas_datareader.data as web
import datetime

tickers = ['AAPL', 'MSFT', 'GOOG', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']
start_date = '2020-01-01'
end_date = '2025-06-30'

os.makedirs('data', exist_ok=True)

df = yf.download(tickers, start=start_date, end=end_date)
df.columns = ['{}_{}'.format(col[0], col[1]) for col in df.columns]  # flatten columns
df = df.reset_index()

df = (
    pd.melt(df, id_vars='Date', var_name='Price_Ticker', value_name='Value')
      .assign(Price_Type=lambda x: x.Price_Ticker.str.split('_').str[0],
              Ticker=lambda x: x.Price_Ticker.str.split('_').str[1])
      .drop(columns='Price_Ticker')
      .pivot_table(index=['Date', 'Ticker'], columns='Price_Type', values='Value')
      .reset_index()
)

df.to_csv('data/combined_tickers.csv', index=False)
print("Saved combined data to yfinance_data/combined_tickers.csv")


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  8 of 8 completed


Saved combined data to yfinance_data/combined_tickers.csv


### Extract and Prepare Mock CSV data from FRED

- Fetch key U.S. economic indicators from the **FRED (Federal Reserve Economic Data)** database (from January 1, 2020 to June 1, 2025).  
- Combine all metrics into a single **DataFrame**.  
- Export to CSV.

In [ ]:
# Define date range
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2025, 6, 1)

# Extract data from FRED
cpi = web.DataReader('CPIAUCSL', 'fred', start_date, end_date)
fed_funds = web.DataReader('FEDFUNDS', 'fred', start_date, end_date)
unemployment = web.DataReader('UNRATE', 'fred', start_date, end_date)
gdp = web.DataReader('GDP', 'fred', start_date, end_date)

# Combine into a single DataFrame
econ_data = cpi.join([fed_funds, unemployment, gdp])
econ_data.columns = ['CPI', 'Federal_Funds_Rate', 'Unemployment_Rate', 'GDP']

# Show data
print(econ_data.head())

# Save to CSV
econ_data.to_csv('data/economic_data_fred.csv')


                CPI  Federal_Funds_Rate  Unemployment_Rate        GDP
DATE                                                                 
2020-01-01  259.127                1.55                3.6  21727.657
2020-02-01  259.250                1.58                3.5        NaN
2020-03-01  258.076                0.65                4.4        NaN
2020-04-01  256.032                0.05               14.8  19935.444
2020-05-01  255.802                0.05               13.2        NaN


### Merge, Clean and Standardize Data 
  - Combine datasets using a **left join** on the date field, unifying column names (`DATE` → `date`).  
  - Drop duplicate date columns and **renames all columns to lowercase** for consistency.  
  - Add a `stock_price_id` for unique identification.  
  - Reorder columns logically, prioritizing ID, date, ticker, and market/economic variables. 

In [ ]:
# Load CSVs
stock_df = pd.read_csv('data/combined_tickers.csv', parse_dates=['Date'])
econ_df = pd.read_csv('data/economic_data_fred.csv', parse_dates=['DATE'])

# Merge and immediately rename date to a unified name
merged_df = pd.merge(
    stock_df,
    econ_df,
    left_on='Date',
    right_on='DATE',
    how='left'
)

# Drop duplicate date column and rename
merged_df = merged_df.drop(columns=['DATE']).rename(columns={'Date': 'date'})

# Optional: add surrogate key
merged_df['stock_price_id'] = merged_df.index + 1

# Reorder columns using lowercase, standardized names
merged_df = merged_df.rename(columns={
    'Ticker': 'ticker',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'CPI': 'cpi_value',
    'Federal_Funds_Rate': 'interest_rate'
})

# Final column arrangement
fact_stock_prices = merged_df[
    ['stock_price_id', 'date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'cpi_value', 'interest_rate']
]

# Preview
print(fact_stock_prices.head())


   stock_price_id       date ticker        open        high         low  \
0               1 2020-01-02   AAPL   71.627100   72.681296   71.373226   
1               2 2020-01-02   AMZN   93.750000   94.900497   93.207497   
2               3 2020-01-02   GOOG   66.681136   68.002778   66.681136   
3               4 2020-01-02  GOOGL   67.018562   68.026016   66.923134   
4               5 2020-01-02   META  205.780144  208.805877  205.302400   

        close       volume  cpi_value  interest_rate  
0   72.620850  135480400.0        NaN            NaN  
1   94.900497   80580000.0        NaN            NaN  
2   67.964508   28132000.0        NaN            NaN  
3   68.026016   27278000.0        NaN            NaN  
4  208.795929   12077100.0        NaN            NaN  


### Creates a time dimension table (`dim_date`) for data warehousing/analysis
  - Covers every day from **January 1, 2020**, to **December 31, 2025**.  
  - Basic: `year`, `month`, `day`.  
  - Temporal: `weekday` (name), `quarter` (1-4).  
  - Each row represents a **unique date** with derived temporal features.

In [6]:
# Define the range of dates
date_range = pd.date_range(start='2020-01-01', end='2025-12-31')

# Create the dimension table
dim_date = pd.DataFrame({
    'date': date_range,
    'year': date_range.year,
    'month': date_range.month,
    'day': date_range.day,
    'weekday': date_range.day_name(),
    'quarter': date_range.quarter
})

# Show preview
print(dim_date.head())


        date  year  month  day    weekday  quarter
0 2020-01-01  2020      1    1  Wednesday        1
1 2020-01-02  2020      1    2   Thursday        1
2 2020-01-03  2020      1    3     Friday        1
3 2020-01-04  2020      1    4   Saturday        1
4 2020-01-05  2020      1    5     Sunday        1
